<a href="https://colab.research.google.com/github/carlos-aular-ds/labo2025v/blob/main/src/ensembles/420_ArbolesAzarosos_BayesianOpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4. Ensembles de Arboles de Decision

Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/labo1"
mkdir -p "/content/buckets"
ln -s "/content/.drive/My Drive/labo1" /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets



archivo_origen="https://storage.googleapis.com/open-courses/austral2025-af91/dataset_pequeno.csv"
archivo_destino="/content/datasets/dataset_pequeno.csv"
archivo_destino_bucket="/content/buckets/b1/datasets/dataset_pequeno.csv"

if ! test -f $archivo_destino_bucket; then
  wget  $archivo_origen  -O $archivo_destino_bucket
fi


if ! test -f $archivo_destino; then
  cp  $archivo_destino_bucket  $archivo_destino
fi



---



## 4.02 Arboles Azarosos con Optimización Bayesiana

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

**NUEVO:** En esta versión se agrega optimización bayesiana para encontrar los mejores hiperparámetros antes del entrenamiento final.

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> **MODIFICADO:** Ahora el porcentaje de campos se optimiza automáticamente en vez de ser fijo al 50%

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Tue Oct 28 05:59:34 PM 2025"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,660381,35.3,1454477,77.7,1454477,77.7
Vcells,1226627,9.4,8388608,64.0,1975128,15.1


In [4]:
# Instalar si es necesario
if (!require("ParBayesianOptimization")) {
  install.packages("ParBayesianOptimization",
                   repos = "https://cloud.r-project.org/",
                   dependencies = TRUE)
}

# Cargar las librerías que necesito
require("data.table")
require("rpart")
require("ParBayesianOptimization")

cat("Todas las librerías cargadas correctamente\n")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘rbibutils’, ‘Deriv’, ‘microbenchmark’, ‘Rdpack’, ‘numDeriv’, ‘doBy’, ‘SparseM’, ‘MatrixModels’, ‘minqa’, ‘nloptr’, ‘reformulas’, ‘RcppEigen’, ‘carData’, ‘abind’, ‘Formula’, ‘pbkrtest’, ‘quantreg’, ‘lme4’, ‘corrplot’, ‘car’, ‘iterators’, ‘ggrepel’, ‘ggsci’, ‘cowplot’, ‘ggsignif’, ‘gridExtra’, ‘polynom’, ‘rstatix’, ‘DiceKriging’, ‘foreach’, ‘dbscan’, ‘lhs’, ‘ggpubr’


Warning message in install.packages("ParBayesianOptimization"):
“installation of package ‘reformulas’ had non-zero exit status”
Warning message in install.packages("ParBayesianOptimization"):
“installation of package ‘lme4’ had non-zero exit status”
Warning message in install.packages("ParBayesianOptimization"):
“installation of package ‘pbkrtest’ had non-zero exit status”
Warning message in install.packages("ParBayesianOptimization"):
“installation of package ‘car’ had non-zero exit status”
Warning message i

Aqui debe cargar SU semilla primigenia

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 111119

# MODIFICADO: Estos parámetros ahora se optimizarán automáticamente
# Los valores iniciales se usarán solo como referencia
# feature_fraction, cp, minsplit, minbucket, maxdepth serán optimizados

PARAM$rpart <- list()  # Se llenará después de la optimización

# voy a generar 512 arboles,
#  a mas arboles mas tiempo de proceso y MEJOR MODELO,
#  pero ganancias marginales
PARAM$num_trees_max <- 10

# NUEVO: Parámetros para la optimización bayesiana
PARAM$bayesopt <- list()
PARAM$bayesopt$init_points <- 10  # evaluaciones aleatorias iniciales
PARAM$bayesopt$iterations <- 20   # iteraciones de optimización
PARAM$bayesopt$trees_validation <- 10  # árboles para validación (menos que final)

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")

# MODIFICADO: Nuevo nombre de experimento para diferenciarlo
experimento <- "exp4020_bayesopt"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

In [ ]:
# NUEVO: Separar dtrain en entrenamiento y validación para la optimización
set.seed(PARAM$semilla_primigenia)
indices_train <- sample(1:nrow(dtrain), 0.7 * nrow(dtrain))

dtrain_opt <- dtrain[indices_train]   # Para optimización (70%)
dvalidacion <- dtrain[-indices_train]  # Para validación (30%)

cat("Registros para optimización:", nrow(dtrain_opt), "\n")
cat("Registros para validación:", nrow(dvalidacion), "\n")

In [ ]:
# Establezco cuales son los campos que puedo usar para la prediccion
# el copy() es por la Lazy Evaluation
campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

### NUEVA SECCIÓN: Definición de Función Objetivo para Optimización Bayesiana

In [ ]:
# NUEVO: Función objetivo que evalúa un conjunto de hiperparámetros
# Retorna la ganancia obtenida en el conjunto de validación
funcion_objetivo <- function(feature_fraction, cp, minsplit,
                             minbucket, maxdepth) {

  # Preparar tabla para acumular predicciones
  tb_pred_val <- dvalidacion[, list(numero_de_cliente, clase_ternaria)]
  tb_pred_val[, prob_acum := 0]

  # Entrenar ensemble con menos árboles para acelerar la optimización
  for (arbolito in 1:PARAM$bayesopt$trees_validation) {

    # Seleccionar campos al azar según feature_fraction
    qty_campos <- as.integer(length(campos_buenos) * feature_fraction)
    campos_random <- sample(campos_buenos, qty_campos)
    campos_random <- paste(campos_random, collapse = " + ")
    formulita <- paste0("clase_ternaria ~ ", campos_random)

    # Entrenar árbol con los hiperparámetros candidatos
    modelo <- rpart(formulita,
      data = dtrain_opt,
      xval = 0,
      control = list(
        cp = cp,
        minsplit = as.integer(minsplit),
        minbucket = as.integer(minbucket),
        maxdepth = as.integer(maxdepth)
      )
    )

    # Predecir y acumular probabilidades
    prediccion <- predict(modelo, dvalidacion, type = "prob")
    tb_pred_val[, prob_acum := prob_acum + prediccion[, "BAJA+2"]]
  }

  # Calcular probabilidad promedio
  tb_pred_val[, prob_promedio := prob_acum / PARAM$bayesopt$trees_validation]

  # Ordenar por probabilidad y seleccionar top 11000
  setorder(tb_pred_val, -prob_promedio)
  tb_pred_val[, Predicted := 0]
  tb_pred_val[1:11000, Predicted := 1]

  # Calcular ganancia
  ganancia <- tb_pred_val[, sum(ifelse(clase_ternaria == "BAJA+2" &
                                        Predicted == 1, 273000,
                                 ifelse(Predicted == 1, -7000, 0)))]

  # La función debe retornar una lista con Score
  return(list(Score = ganancia))
}

cat("Función objetivo definida correctamente\n")

### NUEVA SECCIÓN: Ejecución de Optimización Bayesiana

In [ ]:
# NUEVO: Definir rangos de búsqueda para cada hiperparámetro
bounds <- list(
  feature_fraction = c(0.3, 0.8),    # entre 30% y 80% de features
  cp = c(-1, 0),                      # complexity parameter
  minsplit = c(20, 200),              # mínimo de observaciones para split
  minbucket = c(10, 100),             # mínimo de observaciones en hoja
  maxdepth = c(4, 12)                 # profundidad máxima del árbol
)

cat("Rangos de búsqueda definidos:\n")
print(bounds)

In [ ]:
# NUEVO: Ejecutar optimización bayesiana
cat("\n==========================================================\n")
cat("INICIANDO OPTIMIZACIÓN BAYESIANA DE HIPERPARÁMETROS\n")
cat("==========================================================\n")
cat("Evaluaciones iniciales aleatorias:", PARAM$bayesopt$init_points, "\n")
cat("Iteraciones de optimización:", PARAM$bayesopt$iterations, "\n")
cat("Árboles por evaluación:", PARAM$bayesopt$trees_validation, "\n")
cat("Tiempo estimado: ~30-40 minutos\n")
cat("==========================================================\n\n")

set.seed(PARAM$semilla_primigenia)

resultados_bayes <- bayesOpt(
  FUN = funcion_objetivo,
  bounds = bounds,
  initPoints = PARAM$bayesopt$init_points,
  iters.n = PARAM$bayesopt$iterations,
  verbose = 1
)

cat("\n==========================================================\n")
cat("OPTIMIZACIÓN COMPLETADA\n")
cat("==========================================================\n")

In [ ]:
# NUEVO: Extraer y mostrar los mejores hiperparámetros encontrados
mejores_params <- getBestPars(resultados_bayes)

cat("\n==========================================================\n")
cat("MEJORES HIPERPARÁMETROS ENCONTRADOS\n")
cat("==========================================================\n")
print(mejores_params)
cat("\nGanancia obtenida en validación:", max(resultados_bayes$scoreSummary$Score), "\n")
cat("==========================================================\n\n")

In [ ]:
# NUEVO: Guardar historial de optimización
fwrite(resultados_bayes$scoreSummary,
       file = "bayesopt_history.csv",
       sep = ",")

cat("Historial de optimización guardado en: bayesopt_history.csv\n")

In [ ]:
# NUEVO: Asignar mejores hiperparámetros a PARAM para el entrenamiento final
PARAM$feature_fraction <- mejores_params$feature_fraction
PARAM$rpart$cp <- mejores_params$cp
PARAM$rpart$minsplit <- as.integer(mejores_params$minsplit)
PARAM$rpart$minbucket <- as.integer(mejores_params$minbucket)
PARAM$rpart$maxdepth <- as.integer(mejores_params$maxdepth)

cat("\nParámetros configurados para entrenamiento final:\n")
cat("feature_fraction:", PARAM$feature_fraction, "\n")
cat("cp:", PARAM$rpart$cp, "\n")
cat("minsplit:", PARAM$rpart$minsplit, "\n")
cat("minbucket:", PARAM$rpart$minbucket, "\n")
cat("maxdepth:", PARAM$rpart$maxdepth, "\n")

### Entrenamiento Final con Mejores Hiperparámetros

In [ ]:
# que tamanos de ensemble grabo a disco
grabar <- c(1, 2, 4, 8, 16, 32, 64, 128, 256, 384, 512)

In [ ]:
tb_prediccion <- dfuture[, list(numero_de_cliente)]
# aqui se va acumulando la probabilidad del ensemble
tb_prediccion[, prob_acumulada := 0]

In [ ]:
# MODIFICADO: Ahora usamos TODO el dtrain (sin división) para el entrenamiento final
set.seed(PARAM$semilla_primigenia) # Establezco la semilla aleatoria

cat("\n==========================================================\n")
cat("INICIANDO ENTRENAMIENTO FINAL CON", PARAM$num_trees_max, "ÁRBOLES\n")
cat("Usando HIPERPARÁMETROS OPTIMIZADOS\n")
cat("==========================================================\n\n")

In [ ]:
# MODIFICADO: El loop es el mismo pero ahora usa los hiperparámetros optimizados
for (arbolito in seq(PARAM$num_trees_max) ) {
  message( arbolito, " ")

  # MODIFICADO: Ahora usa PARAM$feature_fraction optimizado
  qty_campos_a_utilizar <- as.integer(length(campos_buenos)
    * PARAM$feature_fraction)

  # elijo los campos al azar
  campos_random <- sample(campos_buenos, qty_campos_a_utilizar)

  # paso de un vector a un string con los elementos
  # separados por un signo de "+"
  # este hace falta para la formula
  campos_random <- paste(campos_random, collapse= " + ")

  # armo la formula para rpart
  formulita <- paste0("clase_ternaria ~ ", campos_random)

  # MODIFICADO: genero el arbol con hiperparámetros optimizados en PARAM$rpart
  # MODIFICADO: Ahora entrena con TODO dtrain (sin split)
  modelo <- rpart(formulita,
    data= dtrain,
    xval= 0,
    control= PARAM$rpart
  )

  # aplico el modelo a los datos que no tienen clase
  prediccion <- predict(modelo, dfuture, type= "prob")

  tb_prediccion[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]

  if (arbolito %in% grabar) {
    umbral_corte <- (1 / 40) * arbolito
    tb_prediccion[, Predicted := as.numeric(prob_acumulada > umbral_corte)]

    # MODIFICADO: Nuevo nombre de archivo para diferenciar de versión sin optimización
    archivo_kaggle <- paste0(
        "KA420_BO_",  # BO = Bayesian Optimization
        sprintf("%.3d", arbolito), # para que tenga ceros adelante
        ".csv"
      )

    # grabo el archivo
    fwrite( tb_prediccion[, list(numero_de_cliente, Predicted)],
      file= archivo_kaggle,
      sep= ","
    )

    # subida a Kaggle
    comando <- "kaggle competitions submit"
    competencia <- "-c labo-i-2025-ba-analista-sr"
    arch <- paste( "-f", archivo_kaggle)

    # MODIFICADO: Mensaje ahora incluye que usó Bayesian Optimization
    mensaje <- paste0("-m 'BayesOpt: cp=", PARAM$rpart$cp,
                     " minsplit=", PARAM$rpart$minsplit,
                     " minbucket=", PARAM$rpart$minbucket,
                     " maxdepth=", PARAM$rpart$maxdepth,
                     " ff=", round(PARAM$feature_fraction, 3), "'" )
    linea <- paste( comando, competencia, arch, mensaje)
    salida <- system(linea, intern=TRUE)
    cat(salida)
  }
}

cat("\n==========================================================\n")
cat("ENTRENAMIENTO FINAL COMPLETADO\n")
cat("==========================================================\n")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

### NUEVA SECCIÓN: Resumen Final

In [ ]:
# NUEVO: Guardar resumen de la ejecución
resumen <- list(
  experimento = experimento,
  semilla = PARAM$semilla_primigenia,
  num_arboles = PARAM$num_trees_max,
  feature_fraction = PARAM$feature_fraction,
  cp = PARAM$rpart$cp,
  minsplit = PARAM$rpart$minsplit,
  minbucket = PARAM$rpart$minbucket,
  maxdepth = PARAM$rpart$maxdepth,
  ganancia_validacion = max(resultados_bayes$scoreSummary$Score),
  evaluaciones_bayesopt = PARAM$bayesopt$init_points + PARAM$bayesopt$iterations
)

# Convertir a data.table y guardar
dt_resumen <- as.data.table(resumen)
fwrite(dt_resumen, file = "resumen_ejecucion.csv", sep = ",")

cat("\n==========================================================\n")
cat("RESUMEN DE LA EJECUCIÓN\n")
cat("==========================================================\n")
print(resumen)
cat("\nResumen guardado en: resumen_ejecucion.csv\n")
cat("Historial completo en: bayesopt_history.csv\n")
cat("==========================================================\n")



---

